# 30E — Source-balance v2 continuous pose from user-controlled video files

**CPU → Run all. No YouTube request is made.** Put the seven source videos in `MyDrive/cALorie_source_balance_v2/videos/` before running, or select the missing files in the upload widget. Each filename must contain its pinned video ID (for example `Sa0sBKUiOvU.mp4`). The notebook verifies duration and records a SHA-256 for every source, extracts continuous 15-FPS MediaPipe pose for the 18 retained windows, and exports no raw video. All outputs remain independently unreviewed and training-disabled.

Required IDs: `Sa0sBKUiOvU`, `nixaLRF0Jd8`, `-HusJ8P4k-4`, `2W9voBSJzP8`, `-iEDHNFNh7Q`, `2AiC7fQyOB8`, `dFzUjzfih7k`.


In [ ]:
%pip install -q mediapipe==0.10.35 opencv-python-headless pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib,json,shutil,urllib.request
import cv2,mediapipe as mp,numpy as np,pandas as pd
from google.colab import drive,files
from IPython.display import display

PINNED_COMMIT='f3aa06c6acb963ed4f2fe2e9fa23d8c11ad02752'
RAW=f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/'
INPUTS={
 'ml/data/source_balance_v2/pose_audit_targets.csv':'480319409c2b6e1fd80e786e22b56ce3bab9ab0d35cba0248494a01d8e4885e4',
 'ml/data/source_balance_v2/pose_visual_triage.csv':'c1b7bf60fda6b957720a0a5feeedab58633f59f3e7e983dce3bf33acb53d3429',
}
WORK=Path('/content/calorie_source_balance_v2_continuous_pose_work')
OUTPUT=Path('/content/calorie_source_balance_v2_continuous_pose')
if WORK.exists():shutil.rmtree(WORK)
if OUTPUT.exists():shutil.rmtree(OUTPUT)
for folder in (WORK,WORK/'inputs',WORK/'uploads',OUTPUT,OUTPUT/'continuous_landmarks',OUTPUT/'pose_contact_sheets'):
 folder.mkdir(parents=True,exist_ok=True)

def sha256(path):
 digest=hashlib.sha256()
 with Path(path).open('rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''):digest.update(block)
 return digest.hexdigest()

def fetch(relative,expected):
 path=WORK/'inputs'/Path(relative).name
 urllib.request.urlretrieve(RAW+relative,path)
 assert sha256(path)==expected,(relative,sha256(path))
 return path

paths={name:fetch(name,digest) for name,digest in INPUTS.items()}
targets=pd.read_csv(paths['ml/data/source_balance_v2/pose_audit_targets.csv'])
triage=pd.read_csv(paths['ml/data/source_balance_v2/pose_visual_triage.csv'])
assert len(targets)==18 and targets.window_id.is_unique
assert set(targets.window_id)==set(triage.window_id)
assert (triage.decision=='retain_for_independent_review_and_source_acquisition').all()
assert (~targets.independent_human_review_complete).all() and (~targets.training_allowed).all()
REQUIRED_IDS=sorted(targets.video_id.unique())
print('Pinned windows:',len(targets),'across',len(REQUIRED_IDS),'sources')
display(triage.groupby(['target_class','coarse_pose_tier']).size().rename('windows').reset_index())


In [ ]:
DRIVE_DIR=Path('/content/drive/MyDrive/cALorie_source_balance_v2/videos')
VIDEO_EXTS={'.mp4','.mkv','.mov','.avi','.webm','.m4v'}
drive.mount('/content/drive',force_remount=False)
DRIVE_DIR.mkdir(parents=True,exist_ok=True)

def discover_sources():
 candidates=[]
 for root in (DRIVE_DIR,WORK/'uploads'):
  if root.exists():
   candidates.extend(p for p in root.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
 found={}
 ambiguous={}
 for video_id in REQUIRED_IDS:
  matches=sorted({p.resolve() for p in candidates if video_id in p.stem})
  if len(matches)==1:found[video_id]=matches[0]
  elif len(matches)>1:ambiguous[video_id]=matches
 return found,ambiguous

source_paths,ambiguous=discover_sources()
assert not ambiguous,'Multiple matching files: '+repr(ambiguous)
missing=[video_id for video_id in REQUIRED_IDS if video_id not in source_paths]
if missing:
 print('Select the missing source files. Their filenames must contain these IDs:')
 print('\n'.join(missing))
 uploaded=files.upload()
 for name,data in uploaded.items():
  destination=WORK/'uploads'/Path(name).name
  destination.write_bytes(data)
 source_paths,ambiguous=discover_sources()
 assert not ambiguous,'Multiple matching files after upload: '+repr(ambiguous)
 missing=[video_id for video_id in REQUIRED_IDS if video_id not in source_paths]
assert not missing,'Still missing files whose names contain: '+', '.join(missing)

def probe(path):
 cap=cv2.VideoCapture(str(path))
 fps=float(cap.get(cv2.CAP_PROP_FPS));frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
 width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
 cap.release()
 assert fps>0 and frames>0 and width>0 and height>0,path
 return {'fps':fps,'frame_count':frames,'duration_s':frames/fps,'width':width,'height':height}

inventory=[]
for video_id,path in source_paths.items():
 metadata=probe(path)
 required_end=float(targets.loc[targets.video_id==video_id,'end_s'].max())
 assert metadata['duration_s']>=required_end-2,(video_id,metadata['duration_s'],required_end)
 inventory.append({'video_id':video_id,'filename':path.name,'sha256':sha256(path),
  **metadata,'required_end_s':required_end,'source_identity_attested':False,
  'raw_video_exported':False,'training_allowed':False})
inventory=pd.DataFrame(inventory).sort_values('video_id')
source_hashes=dict(zip(inventory.video_id,inventory.sha256))
inventory.to_csv(OUTPUT/'source_inventory.csv',index=False)
print('Source files ready. Hashing and duration checks passed.')
display(inventory[['video_id','filename','duration_s','fps','width','height','required_end_s']])


In [ ]:
MODEL=WORK/'pose_landmarker_full.task'
MODEL_URL=('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
 'pose_landmarker_full/float16/latest/pose_landmarker_full.task')
urllib.request.urlretrieve(MODEL_URL,MODEL)
assert MODEL.stat().st_size>1_000_000
TARGET_FPS=15.0
KEEP_17=[0,2,5,7,8,11,12,13,14,15,16,23,24,25,26,27,28]
CRITICAL=[11,12,23,24,25,26,27,28]
SKELETON=[(11,13),(13,15),(12,14),(14,16),(11,12),(11,23),(12,24),(23,24),(23,25),(25,27),(24,26),(26,28)]

def normalize_pose(frame17):
 xyz=frame17[:,:3].copy();hip=(xyz[11]+xyz[12])/2;shoulder=(xyz[5]+xyz[6])/2
 scale=max(float(np.linalg.norm(xyz[5]-xyz[6])),float(np.linalg.norm(shoulder-hip)),1e-6)
 return np.concatenate([(xyz-hip)/scale,frame17[:,3:4]],axis=1)

def draw_overlay(frame,points,timestamp):
 tile=cv2.resize(frame,(320,180))
 if points:
  for a,b in SKELETON:
   if min(points[a].visibility,points[b].visibility)>=.5:
    cv2.line(tile,(int(points[a].x*320),int(points[a].y*180)),(int(points[b].x*320),int(points[b].y*180)),(0,255,0),2)
  for point in points:
   if point.visibility>=.5:
    cv2.circle(tile,(int(point.x*320),int(point.y*180)),2,(0,0,255),-1)
 cv2.rectangle(tile,(0,0),(112,25),(0,0,0),-1)
 cv2.putText(tile,f'{int(timestamp//60):02d}:{timestamp%60:04.1f}',(4,18),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,255),1)
 return tile

def sampled_frames(path,start_s,end_s):
 cap=cv2.VideoCapture(str(path));fps=float(cap.get(cv2.CAP_PROP_FPS))
 cap.set(cv2.CAP_PROP_POS_MSEC,max(0,start_s-.5)*1000)
 next_sample=start_s
 while True:
  ok,frame=cap.read()
  if not ok:break
  timestamp=float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000
  if timestamp<=0:
   timestamp=float(cap.get(cv2.CAP_PROP_POS_FRAMES))/fps
  if timestamp>=end_s:break
  if timestamp+1e-4<next_sample:continue
  yield timestamp,frame
  while next_sample<=timestamp+1e-4:next_sample+=1/TARGET_FPS
 cap.release()

def audit_window(row,landmarker):
 raw=[];normalized=[];detected=[];timestamps=[];overlays=[]
 for frame_index,(timestamp,frame) in enumerate(sampled_frames(source_paths[row.video_id],float(row.start_s),float(row.end_s))):
  rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
  result=landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
  points=result.pose_landmarks[0] if result.pose_landmarks else None
  if points:
   full=np.asarray([[p.x,p.y,p.z,float(p.visibility or 0)] for p in points],np.float32)
   raw.append(full);normalized.append(normalize_pose(full[KEEP_17]));detected.append(True)
  else:
   raw.append(np.full((33,4),np.nan,np.float32));normalized.append(np.full((17,4),np.nan,np.float32));detected.append(False)
  timestamps.append(timestamp)
  if frame_index%15==0:overlays.append(draw_overlay(frame,points,timestamp))
 assert len(timestamps)>=max(1,int((float(row.end_s)-float(row.start_s))*TARGET_FPS*.8)),(row.window_id,len(timestamps))
 raw=np.stack(raw);normalized=np.stack(normalized);detected=np.asarray(detected,bool);visibility=raw[:,:,3]
 np.savez_compressed(OUTPUT/'continuous_landmarks'/(row.window_id+'.npz'),
  image_landmarks=raw,landmarks=normalized,timestamps_s=np.asarray(timestamps,np.float32),detected=detected,
  target_fps=np.float32(TARGET_FPS),video_id=row.video_id,target_class=row.target_class,
  group_id=f'source_balance_v2:{row.video_id}',source_sha256=source_hashes[row.video_id],
  source_identity_attested=False,training_allowed=False,not_training_landmarks=True)
 page_root=OUTPUT/'pose_contact_sheets'/row.window_id;page_root.mkdir(parents=True,exist_ok=True)
 for page_index,offset in enumerate(range(0,len(overlays),12)):
  tiles=overlays[offset:offset+12]
  while len(tiles)<12:tiles.append(np.zeros_like(tiles[0]))
  page=np.vstack([np.hstack(tiles[i:i+4]) for i in (0,4,8)])
  cv2.imwrite(str(page_root/f'{page_index:03d}.jpg'),page,[cv2.IMWRITE_JPEG_QUALITY,86])
 joint_rates=np.nanmean(visibility>=.5,axis=0) if detected.any() else np.zeros(33)
 full_body=np.all(visibility[:,CRITICAL]>=.5,axis=1) if detected.any() else np.zeros(len(raw),bool)
 return {'frames':len(timestamps),'detection_rate':float(detected.mean()),
  'visible_joint_rate_33':float(np.nanmean(visibility>=.5)) if detected.any() else 0.0,
  'full_body_frame_rate':float(full_body.mean()),'left_ankle_visible_rate':float(joint_rates[27]),
  'right_ankle_visible_rate':float(joint_rates[28]),'pose_pages':len(list(page_root.glob('*.jpg'))),
  'landmark_path':f'continuous_landmarks/{row.window_id}.npz'}


In [ ]:
options=mp.tasks.vision.PoseLandmarkerOptions(
 base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),
 running_mode=mp.tasks.vision.RunningMode.IMAGE,num_poses=1,
 min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,
 output_segmentation_masks=False)
rows=[]
with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
 for row in targets.sort_values(['video_id','start_s']).itertuples(index=False):
  record=row._asdict();record.update(status='pose_failed',error='')
  try:record.update(audit_window(row,landmarker),status='pose_ready')
  except Exception as exc:record['error']=f'{type(exc).__name__}: {exc}'[:500]
  rows.append(record)
  print(row.window_id,record['status'],record.get('frames',0),f"{record.get('detection_rate',0):.3f}")
quality=pd.DataFrame(rows)
quality['review_status']='draft';quality['independent_human_review_complete']=False;quality['training_allowed']=False
quality.to_csv(OUTPUT/'pose_quality.csv',index=False)
targets.to_csv(OUTPUT/'candidate_windows.csv',index=False)
triage.to_csv(OUTPUT/'coarse_pose_triage.csv',index=False)
metadata={'notebook':'30e_colab_source_balance_v2_continuous_pose_from_upload.ipynb',
 'mode':'user_controlled_source_files','runtime':'CPU','pinned_commit':PINNED_COMMIT,
 'target_fps':TARGET_FPS,'window_count':len(targets),'pose_ready':int((quality.status=='pose_ready').sum()),
 'source_count':len(source_paths),'youtube_requests':0,'raw_video_exported':False,
 'source_identity_attested':False,'training_authorized_landmarks_exported':False,
 'independent_human_review_complete':False,'training_allowed':False,
 'next_step':'verify source identity, review continuous overlays and exact motion boundaries independently'}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
checks=[]
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file() and p.name!='SHA256SUMS.txt'):
 checks.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(checks)+'\n')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
display(quality[['window_id','target_class','status','frames','detection_rate','visible_joint_rate_33','full_body_frame_rate']])
print('Created:',archive,round(Path(archive).stat().st_size/1e6,1),'MB')
files.download(archive)


Return `calorie_source_balance_v2_continuous_pose.zip` for verification. A `pose_ready` row still is not a reviewed label. Source identity is deliberately left unattested in the machine output; confirm each file independently, review exact motion boundaries, and keep `training_allowed=false` until that review is complete.
